# Fermi/SDSS-V Blazar Analysis Pipeline — Demo

This notebook demonstrates how to load pre-computed spectral fitting results from the cache and reproduce all diagnostic plots.

### Before you start
1. Download the cache from Zenodo: DOI to be added upon publication
2. Place Fits_with_Native_resampling_cache/ in the same directory as this notebook
3. Install dependencies: pip install -r requirements.txt

### Reference
Nlowie et al. (in prep.)

In [ ]:
import sys
sys.path.insert(0, "../plotting")

from plot_from_cache import (
    RedshiftResultsCache,
    plot_from_cache,
    plot_multi_object_comparison_single,
    compute_EW_for_all_lines,
)
import numpy as np
from astropy.table import Table
from astropy.io import fits

print("Imports successful.")

## 1. Initialise the cache

In [ ]:
CACHE_DIR = "Fits_with_Native_resampling_cache"
cache = RedshiftResultsCache(cache_dir=CACHE_DIR)
cached_files = cache.list_cached_objects()

## 2. Load and plot a single source

Each source is identified by its SDSS_ID and MJD.
The cache filename format is obj_{SDSS_ID}_{MJD}.pkl.gz.
Check the cache directory for the exact filename to find the MJD for each source.

In [ ]:
# Replace with any SDSS_ID and MJD from the cache
SDSS_ID = "your_sdss_id_here"
MJD     = 0  # replace with actual MJD

results = cache.load_object_results(SDSS_ID, mjd=MJD)

meta  = results["metadata"]
lmfit = results["lmfit_results"]
print(f"SDSS_ID:     {meta[chr(39)]SDSS_ID[chr(39)]}")
print(f"Fermi class: {meta[chr(39)]fermi_class[chr(39)]}")
print(f"Best model:  {lmfit[chr(39)]best_label[chr(39)]}")
print(f"z_fit:       {lmfit[chr(39)]z_best[chr(39)]:.4f}")

In [ ]:
fig1, fig2 = plot_from_cache(results, save_dir=None)

## 3. Inspect equivalent widths

In [ ]:
spec   = results["spectrum"]
z_best = results["lmfit_results"]["z_best"]

results_ew = compute_EW_for_all_lines(
    spec["common_wave"], spec["flux_resamp"],
    spec["err_resamp"],  spec["fit_mask"], z_best)

print(f"Equivalent widths at z={z_best:.4f}:")
for name, obs, ew, ew_err, snr, detected, ltype, window in results_ew:
    if detected:
        print(f"  {name:<15} {ltype:<12} EW={ew:.2f} +/- {ew_err:.2f} A  S/N={snr:.1f}")

## 4. Access raw fit parameters

In [ ]:
lmfit = results["lmfit_results"]
print(f"Best model:  {lmfit[chr(39)]best_label[chr(39)]}")
print(f"z_fit:       {lmfit[chr(39)]z_best[chr(39)]:.4f}")
print(f"chi2:        {lmfit[chr(39)]best_fit_params[chr(39)][chr(39)]chisqr[chr(39)]:.2f}")
print(f"redchi:      {lmfit[chr(39)]best_fit_params[chr(39)][chr(39)]redchi[chr(39)]:.3f}")
print(f"AICc margin: {lmfit[chr(39)]aicc_margin[chr(39)]:.1f}")
print(f"PL alpha:    {lmfit[chr(39)]pl_alpha[chr(39)]:.4f}")
print(f"PL delta:    {lmfit[chr(39)]pl_delta[chr(39)]:.4f}")